In [ ]:
# ============================================================
#  screening_diagnostics — GOOGLE COLAB edition (self-contained)
#  Paste this whole thing into ONE Colab cell and run. It:
#    1. mounts Google Drive
#    2. finds your parquet by name (no fragile hardcoded path)
#    3. defines the diagnostics functions inline (no upload/import needed)
#    4. runs and writes diag_*.csv next to your data
#
#  Change FILENAME below to "combined.parquet" if you want the raw pull instead
#  of the deduped canonical set.
# ============================================================

from __future__ import annotations
import os, re, glob
import pandas as pd

# ---- 1. mount + locate the file ----
from google.colab import drive
drive.mount('/content/drive')

FILENAME = "master_canonical.parquet"          # or "combined.parquet"
hits = glob.glob(f"/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/{FILENAME}", recursive=True)
assert hits, (f"{FILENAME} not found under /content/drive. Is Drive synced? "
              f"Open the Files panel and confirm /content/drive is populated.")
PATH   = hits[0]
OUTDIR = os.path.join(os.path.dirname(PATH), "screening_diag")
print("Using :", PATH)
print("Output:", OUTDIR)

# ---- 2. lexicons ----
SEP = " | "
GENERA = ["ixodes","amblyomma","rhipicephalus","boophilus","dermacentor","haemaphysalis",
          "hyalomma","ornithodoros","argas","otobius","carios","nuttalliella","rhipicentor",
          "margaropus","anomalohimalaya","cosmiomma","nosomma"]
FAMILIES = ["ixodidae","argasidae","ixodida","nuttalliellidae"]
SPECIFIC = GENERA + FAMILIES + ["tick-borne","tick borne","tickborne"]
_specific_re = re.compile(r"\b(?:" + "|".join(map(re.escape, SPECIFIC)) + r")\b", re.IGNORECASE)
_generic_re  = re.compile(r"\btick(?:s)?\b|\bacari\b", re.IGNORECASE)
_mesh_re     = re.compile(r"tick|ixod|argas|acari|amblyomma|rhipicephalus|dermacentor|hyalomma|haemaphysalis", re.IGNORECASE)


# ---- 3. helpers ----
def _col(df, name):
    return df[name].astype("string").fillna("") if name in df.columns else pd.Series([""]*len(df), index=df.index)


def _text(df, *names):
    out = _col(df, names[0])
    for n in names[1:]:
        out = out.str.cat(_col(df, n), sep=" ")
    return out


def discriminate(df, col, pos_mask, generic_mask, multivalued=False, top=500):
    """Return a table: value, n_total, n_pos, n_generic_only, pos_rate, enrichment."""
    if col not in df.columns:
        return pd.DataFrame()
    base = pos_mask.mean() or 1e-9
    if multivalued:
        s = _col(df, col).str.split(r"\s*\|\s*", regex=True)   # split on ' | ' / '|'
        long = s.explode()
        idx = long.index
        val = long.str.strip()
        keep = val.ne("")
        val, idx = val[keep], idx[keep]
        tot = val.value_counts()
        pos = val[pos_mask.reindex(idx).fillna(False).values].value_counts()
        gen = val[generic_mask.reindex(idx).fillna(False).values].value_counts()
    else:
        v = _col(df, col).str.strip()
        keep = v.ne("")
        tot = v[keep].value_counts()
        pos = v[keep & pos_mask].value_counts()
        gen = v[keep & generic_mask].value_counts()
    out = pd.DataFrame({"n_total": tot})
    out["n_pos"] = pos.reindex(out.index).fillna(0).astype(int)
    out["n_generic_only"] = gen.reindex(out.index).fillna(0).astype(int)
    out["pos_rate"] = (out["n_pos"] / out["n_total"]).round(3)
    out["enrichment"] = (out["pos_rate"] / base).round(2)
    return out.sort_values(["n_pos","enrichment"], ascending=False).head(top)


def run_diagnostics(path, outdir="./screening_diag", top=500):
    os.makedirs(outdir, exist_ok=True)
    df = pd.read_parquet(path)
    n = len(df)
    print(f"Loaded {n:,} rows")

    title_abs = _text(df, "oa_title", "oa_abstract", "oa_keywords")
    mesh      = _col(df, "oa_mesh")

    has_genus   = title_abs.str.contains(_specific_re)
    has_mesh    = mesh.str.contains(_mesh_re)
    pos_mask    = has_genus | has_mesh                      # high-confidence tick
    has_generic = title_abs.str.contains(_generic_re)
    generic_only = has_generic & ~pos_mask                  # ambiguous / homonym-prone

    print(f"  genus/MeSH-positive (high-confidence): {pos_mask.sum():,} ({pos_mask.mean():.1%})")
    print(f"  generic-'tick'-only (ambiguous):       {generic_only.sum():,} ({generic_only.mean():.1%})")
    print(f"  neither (matched via fulltext only):   {(~pos_mask & ~has_generic).sum():,}")
    if "oa_abstract" in df.columns:
        miss = _col(df, "oa_abstract").eq("")
        print(f"  abstract missing overall: {miss.mean():.1%} | within high-conf: {miss[pos_mask].mean():.1%}")

    plan = [("oa_topic", False), ("oa_subfield", False), ("oa_field", False), ("oa_domain", False),
            ("oa_journal", False), ("oa_topics_all", True), ("oa_mesh", True),
            ("oa_concepts", True), ("oa_keywords", True)]
    for col, mv in plan:
        tab = discriminate(df, col, pos_mask, generic_only, multivalued=mv, top=top)
        if tab.empty:
            continue
        tab.to_csv(os.path.join(outdir, f"diag_{col}.csv"))
        print(f"\n=== {col} — top values in the high-confidence tick set ===")
        print(tab.head(15).to_string())
        deny = tab[(tab["n_generic_only"] >= 20) & (tab["pos_rate"] <= 0.05)]
        if not deny.empty:
            print(f"  --- likely HOMONYM values (generic-only, low pos_rate) ---")
            print(deny.head(10)[["n_total","n_generic_only","pos_rate"]].to_string())

    print(f"\nSaved per-column CSVs to {outdir}/  (review, then lift the top rows into allow/deny lists)")
    return df


# ---- 4. run ----
df = run_diagnostics(PATH, outdir=OUTDIR)